## Part 4. Drawbacks of current deep survival analysis solutions

Written by Sundong Kim (sundong.kim@kaist.ac.kr), Jan 19th, 2019,

In this notebook, we introduce some drawbacks of the current deep survival analysis solutions (WSDM'17, AAAI'18, AAAI'19).
* First, we conjecture that those deep survival analysis are very slow compared to traditional machine learning methods.
* Second, we introduce how to make it faster
    * rnn -> cnn
    * dropouts
    * how to handle features
    * how to optimize

In [ ]:
import pandas as pd
import numpy as np
import os
import random

# In this tutorial, we will use a store_A dataset.
pre_release_path = '../data_sample/indoor/store_A/'

In [ ]:
# Load dataset

train_labels = pd.read_csv(pre_release_path+'train_labels.tsv', sep='\t')
test_labels = pd.read_csv(pre_release_path+'test_labels.tsv', sep='\t')
train_visits = pd.read_csv(pre_release_path+'train_visits.tsv', sep='\t')
test_visits = pd.read_csv(pre_release_path+'test_visits.tsv', sep='\t')
wifi_sessions = pd.read_csv(pre_release_path+'wifi_sessions.tsv', sep='\t')

### 1. Exploring dataset

In [ ]:
train_labels.head(5)

In [ ]:
train_visits.head(5)

In [ ]:
test_labels.head(5)

In [ ]:
test_visits.head(5)

In [ ]:
wifi_sessions.head(5)

**Some statistics**

In [ ]:
# Number of unique users

print(len(train_labels),len(train_visits),len(test_labels),len(test_visits),len(wifi_sessions))

In [ ]:
# Number of unique users

len(set(train_visits.wifi_id)), len(set(test_visits.wifi_id)) 

In [ ]:
# Revisit ratio

print(train_labels.revisit_intention.value_counts())
print()
print(test_labels.revisit_intention.value_counts())

In [ ]:
# Revisit interval - train data - among users who revisit
%matplotlib inline
train_labels.revisit_interval.hist()

In [ ]:
# Revisit interval - test data - among users who revisit
test_labels.revisit_interval.hist()

In [ ]:
# Wifi session logs distribution by dwell_time
wifi_sessions.dwell_time.hist(bins=100)

In [ ]:
# Wifi session logs distribution by each month:
#  - The number of logs from train period is larger than the number of logs from test period.
#  - Since we only retain one indoor visit for each wifi-id in the test period.
wifi_sessions.ts.hist(bins=12)

In [ ]:
# Wifi session logs distribution by area
wifi_sessions.area.value_counts()

### 2. Feature generation

In [ ]:
### Before feature engineering, querying some useful information from wifi-sessions data, and add to the dataframe.

def add_infos(df):
    df['l_index'] = df.apply(lambda x: [int(x) for x in x['indices'].split(';')], axis=1)
    df['dwell_times'] = df['l_index'].apply(lambda x: [wifi_sessions[wifi_sessions['index'] == idx]['dwell_time'].values[0] for idx in x])
    df['areas'] = df['l_index'].apply(lambda x: [wifi_sessions[wifi_sessions['index'] == idx]['area'].values[0] for idx in x])
    return df

In [ ]:
train_visits = add_infos(train_visits)
test_visits = add_infos(test_visits)

In [ ]:
train_visits.head(3)

In [ ]:
### Sample code to generate features 

def statistical_feature_generator(x):
    fs = []

    total_dwell_time = sum(x['dwell_times'])   # total dwell time
    num_area_trajectory_have = len(x['dwell_times'])  # the number of area
    num_unique_area_sensed = len(set(x['areas']))  # the number of unique areas
    
    fs.append(total_dwell_time)
    fs.append(num_area_trajectory_have)  
    fs.append(num_unique_area_sensed)     
    
    return fs



def add_statistical_features(train_visits):
    df = train_visits.copy()
    
    features = df.apply(lambda x: statistical_feature_generator(x), axis=1)
    featureName = ['total_dwell_time', 'num_area', 'num_unique_area']
    
    fdf = pd.DataFrame(list(np.asarray(features)), index=features.index, columns = featureName)
    
    # Combine feature values to the dataframe
    df = pd.concat([df, fdf], axis=1)
    del fdf
    
    return df

In [ ]:
train_visits = add_statistical_features(train_visits)
test_visits = add_statistical_features(test_visits)

In [ ]:
train_visits.head(3)

### 3. Revisit Prediction (Binary classification)

In [ ]:
### Retain only feature values

def remove_unnecessary_features(df):
    unnecessary_attributes = ['visit_id', 'wifi_id', 'date', 'indices', 'l_index', 'dwell_times', 'areas']
    all_attributes = list(df.columns)
    for attribute in unnecessary_attributes:
        try:
            all_attributes.remove(attribute)
        except:
            pass
    df = df[all_attributes]
    return df

df_train = remove_unnecessary_features(train_visits)
df_test = remove_unnecessary_features(test_visits)

In [ ]:
import time
import xgboost as xgb
from sklearn import metrics
from sklearn.svm import SVC
from sklearn import linear_model
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def show_intention_classification_result(y_pred, y_test):
    return metrics.accuracy_score(y_test, y_pred)

def show_interval_regression_result(y_pred, y_test):
    return metrics.mean_squared_error(y_test, y_pred)

Downsampling for measuring binary classification accuracy

In [ ]:
df_train = pd.concat([df_train, train_labels['revisit_intention']], axis=1)
df_test = pd.concat([df_test, test_labels['revisit_intention']], axis=1)

In [ ]:
def label_balancing(df, name_target_column):
    ## 1:1 Downsampling
    minimum_label_num = list(df[name_target_column].value_counts())[-1]
    
    df_list = []
    for value in df[name_target_column].unique():
        sub_dfs = df.loc[df[name_target_column] == value]
        new_sub_dfs = sub_dfs.iloc[np.random.permutation(len(sub_dfs))][:minimum_label_num]  ## Random Downsampling according to smallest label size
        df_list.append(new_sub_dfs)
        del sub_dfs
        
    new_df = pd.concat(df_list).sort_index()
    
    return new_df

In [ ]:
df_train.head(3)

In [ ]:
acc = []

print('Class label distribution before downsampling - Train data: revisit_intention 0: {}, 1: {}'.format(
        df_train.revisit_intention.value_counts()[0],
        df_train.revisit_intention.value_counts()[1]))
print('Class label distribution before downsampling - Test data: revisit_intention 0: {}, 1: {}'.format(
        df_test.revisit_intention.value_counts()[0],
        df_test.revisit_intention.value_counts()[1]))
print()
print('-----------   Experiments Begin   -------------')
print()

for i in range(5):    
    
    ## Making downsampled dataset for measuring binary classification accuracy - baseline = 0.5
    whole_balanced_train = label_balancing(df_train, 'revisit_intention') 
    whole_balanced_test = label_balancing(df_test, 'revisit_intention') 
    print('Class label distribution after downsampling - Train data: revisit_intention 0: {}, 1: {}'.format(
        whole_balanced_train.revisit_intention.value_counts()[0],
        whole_balanced_train.revisit_intention.value_counts()[1]))
    print('Class label distribution after downsampling - Test data: revisit_intention 0: {}, 1: {}'.format(
        whole_balanced_test.revisit_intention.value_counts()[0],
        whole_balanced_test.revisit_intention.value_counts()[1]))
    
    
    for (train_data, test_data, ref) in [(whole_balanced_train, whole_balanced_test, 'Downsampled')]:
        train_array = np.asarray(train_data)  
        test_array = np.asarray(test_data)  

        # Dividing features and labels
        X_train, y_train = train_array[:, :-1], train_array[:, -1].astype(int)
        X_test, y_test = test_array[:, :-1], test_array[:, -1].astype(int)

        # Training
        start = time.time()
        clf = Pipeline([
          ('classification', XGBClassifier(max_depth=5, learning_rate=0.1))
        ])
        clf = clf.fit(X_train, y_train)

        # Prediction
        y_pred = clf.predict(X_test)
        clas_rslt = metrics.accuracy_score(y_test, y_pred)
        done = time.time()
        elapsed = done-start

        # Result
        print('Classification result', round(clas_rslt, 4))
#         print('Elapsed time:', round(elapsed, 4))
        print()
        acc.append(clas_rslt)
    
print()
print('-----------   Performance of our model   -------------')
print()
print('Average accuracy: {:.4f}'.format(np.mean(acc)))